# Machine Learning-Based Network Intrusion Detector
**Student:** Trupti Devanand Salunkhe  
**Institute:** Sandip University, Nashik  
**Course:** B.Sc. Forensic Science — Third Year

> **Scope:** This notebook uses synthetic demonstration data. Its metrics are not evidence of real-world performance. For meaningful research, repeat evaluation on a documented public dataset.


## 1. Objective
Demonstrate how a supervised machine-learning model can classify network-flow-like records as normal or suspicious, then evaluate the model using standard classification metrics.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, ConfusionMatrixDisplay


## 2. Create a synthetic demonstration dataset
The features resemble basic network-flow indicators. Labels are generated by the same simulation, so this is a teaching example—not a realistic benchmark.


In [ ]:
def make_demo_data(n=2500, random_state=42):
    rng = np.random.default_rng(random_state)
    suspicious = rng.binomial(1, 0.35, n)
    duration = rng.gamma(shape=2.0 + suspicious * 0.8, scale=1.5, size=n)
    bytes_out = rng.lognormal(mean=7.0 + suspicious * 0.9, sigma=0.8, size=n)
    bytes_in = rng.lognormal(mean=7.2 - suspicious * 0.35, sigma=0.9, size=n)
    packet_count = rng.poisson(lam=18 + suspicious * 22, size=n) + 1
    failed_connections = rng.poisson(lam=0.4 + suspicious * 3.0, size=n)
    unique_ports = np.clip(rng.poisson(lam=2 + suspicious * 8, size=n), 1, 30)
    protocol_code = rng.choice([0, 1, 2], size=n, p=[0.55, 0.35, 0.10])
    return pd.DataFrame({'duration_seconds': duration, 'bytes_out': bytes_out, 'bytes_in': bytes_in, 'packet_count': packet_count, 'failed_connections': failed_connections, 'unique_destination_ports': unique_ports, 'protocol_code': protocol_code, 'label': suspicious})

df = make_demo_data()
df.head()


## 3. Explore and prepare the data
Label `0` means normal in this simulation; label `1` means suspicious. The labels are simulated rather than collected from real network events.


In [ ]:
display(df.describe().round(2))
print('\nClass counts:')
print(df['label'].map({0:'Normal', 1:'Suspicious'}).value_counts())
X = df.drop(columns='label')
y = df['label']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)


## 4. Train the model
Random Forest is an ensemble model that combines many decision trees. It is a reasonable baseline for mixed numeric features.


In [ ]:
model = RandomForestClassifier(n_estimators=150, random_state=42, class_weight='balanced')
model.fit(X_train, y_train)
predictions = model.predict(X_test)
print('Training complete. Test records:', len(y_test))


## 5. Evaluate the model
Accuracy alone can be misleading in imbalanced security data. Precision, recall, F1-score, and the confusion matrix provide additional context.


In [ ]:
metrics = {
    'Accuracy': accuracy_score(y_test, predictions),
    'Precision': precision_score(y_test, predictions, zero_division=0),
    'Recall': recall_score(y_test, predictions, zero_division=0),
    'F1-score': f1_score(y_test, predictions, zero_division=0)
}
pd.Series(metrics, name='Score').round(3)

print(classification_report(y_test, predictions, target_names=['Normal', 'Suspicious'], zero_division=0))
ConfusionMatrixDisplay.from_predictions(y_test, predictions, display_labels=['Normal', 'Suspicious'], cmap='Blues')
plt.title('Confusion Matrix — Synthetic Demonstration')
plt.tight_layout()
plt.show()


## 6. Feature importance
Feature importance indicates which generated features the trained model relied on most. Because the dataset is synthetic, these importances should not be generalized to real networks.


In [ ]:
importance = pd.Series(model.feature_importances_, index=X.columns).sort_values()
importance.plot(kind='barh', title='Feature Importance — Synthetic Demonstration')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()


## 7. Forensic relevance, limitations, and future work
- **Relevance:** network anomaly alerts can help investigators prioritize records for further examination.
- **Limitations:** synthetic data does not capture the complexity of real traffic; false positives and false negatives are possible.
- **Evidence handling:** preserve original logs, record timestamps and acquisition methods, maintain access controls, and document each analytical step.
- **Future work:** test on a documented public dataset such as CIC-IDS2017 or UNSW-NB15; prevent data leakage; report per-class metrics; compare with baseline models; document dataset licensing and provenance.
- **Ethics:** use only traffic data you are authorized to access. This prototype is for education and must not automatically block systems or be used as the sole basis for incident conclusions.
